# California — Insurance Code (INS) → `data/california/ins_codes/*.md`

California’s **Insurance Code** is codified as **INS** on the Legislature’s official site ([Insurance Code — leginfo](https://leginfo.legislature.ca.gov/faces/codesTOCSelected.xhtml?tocCode=INS&tocTitle=Insurance+Code+-+INS)). Section text is served at **`codes_displaySection.xhtml?lawCode=INS&sectionNum=…`** and works well with plain **`httpx`**.

**Discovery:** Justia publishes a browsable mirror under **`/codes/california/<year>/code-ins/`** (e.g. [2024 Insurance Code](https://law.justia.com/codes/california/2024/code-ins/)). That index is behind **Cloudflare**, so this notebook uses **`curl_cffi`** (browser TLS fingerprint) only to **crawl** chapter/part links and collect every **`section-…`** URL, then derives the **section number** for each file.

**Download:** Each section’s **authoritative** HTML is fetched from **leginfo** with **`httpx`**, text is taken from **`#single_law_section`**, and saved as **`INS_sec_<section>.md`** (dots in section numbers become underscores in filenames, e.g. `INS_sec_1871_7.md`).

**Extras:** Optional **`_extras_sections.txt`** in `OUT_DIR` lists extra section numbers (one per line, e.g. `790.03`) if anything is missing from the Justia crawl.

**Config:** **`CODE_YEAR`** (Justia path, default **2024**). **`MAX_SECTIONS`** caps downloads; **`MAX_DISCOVERY_PAGES`** caps TOC pages fetched during the Justia crawl (**0** = no limit). **`REUSE_DISCOVERED_IDS`** skips a repeat crawl when **`_california_ins_section_ids.txt`** exists.

**Politeness:** **`REQUEST_DELAY_SEC`** applies to both Justia (discovery) and leginfo (downloads).

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi httpx beautifulsoup4 certifi


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import quote, urljoin, urlparse

import certifi
import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE_JUSTIA = "https://law.justia.com"
ORIGIN_LEGINFO = "https://leginfo.legislature.ca.gov"

# Justia uses a calendar year in the URL; adjust if you need a different edition.
CODE_YEAR = "2024"
JUSTIA_PREFIX = f"{BASE_JUSTIA}/codes/california/{CODE_YEAR}/code-ins/"

OUT_DIR = Path("data") / "california" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-CA-INS/1.0 (public California Insurance Code; educational indexing)"
CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.25
TIMEOUT = 60.0

# 0 = no cap on leginfo section downloads
MAX_SECTIONS = 0
# 0 = crawl every Justia TOC page under code-ins; set e.g. 80 to smoke-test discovery only
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True
REUSE_DISCOVERED_IDS = True

VERIFY_SSL = True
_verify = certifi.where() if VERIFY_SSL else False

section_path_re = re.compile(r"/section-([\d.]+)/?$", re.I)


In [3]:
def section_sort_key(section_id: str) -> tuple[int, ...]:
    """Sort INS section numbers: 100, 100.1, 1871.7, …"""
    s = section_id.strip().rstrip(".")
    parts: list[int] = []
    for p in s.split("."):
        if p.isdigit():
            parts.append(int(p))
    return tuple(parts) if parts else (0,)


def justia_canon(url: str) -> str:
    u = url.split("#", 1)[0]
    return u if u.endswith("/") else u + "/"


def is_justia_toc_url(url: str) -> bool:
    if not url.startswith(JUSTIA_PREFIX) or url.rstrip("/") + "/" == JUSTIA_PREFIX:
        return False
    if section_path_re.search(urlparse(url).path):
        return False
    return True


def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def discover_section_ids_justia() -> set[str]:
    """BFS Justia code-ins tree; return INS section numbers (no trailing dot)."""
    seen_pages: set[str] = set()
    enqueued: set[str] = {JUSTIA_PREFIX}
    queue: deque[str] = deque([JUSTIA_PREFIX])
    section_ids: set[str] = set()
    while queue:
        page_url = queue.popleft()
        if page_url in seen_pages:
            continue
        if MAX_DISCOVERY_PAGES and len(seen_pages) >= MAX_DISCOVERY_PAGES:
            break
        seen_pages.add(page_url)
        html = curl_get(page_url)
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = justia_canon(urljoin(BASE_JUSTIA, a["href"]))
            if not absu.startswith(JUSTIA_PREFIX):
                continue
            m = section_path_re.search(urlparse(absu).path)
            if m:
                section_ids.add(m.group(1).rstrip("."))
                continue
            if is_justia_toc_url(absu) and absu not in enqueued:
                enqueued.add(absu)
                queue.append(absu)
    print(f"Justia crawl: visited {len(seen_pages)} pages, found {len(section_ids)} section ids")
    return section_ids


def load_extras(path: Path) -> set[str]:
    if not path.exists():
        return set()
    out: set[str] = set()
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        line = line.split("#", 1)[0].strip()
        if not line:
            continue
        out.add(line.rstrip("."))
    return out


def section_id_to_leginfo_num(section_id: str) -> str:
    s = section_id.strip().rstrip(".")
    return s + "."


def leginfo_display_url(section_id: str) -> str:
    sn = section_id_to_leginfo_num(section_id)
    return (
        f"{ORIGIN_LEGINFO}/faces/codes_displaySection.xhtml"
        f"?lawCode=INS&sectionNum={quote(sn, safe='.')}"
    )


def section_id_to_filename(section_id: str) -> str:
    safe = section_id.replace(".", "_")
    return f"INS_sec_{safe}.md"


def extract_leginfo_body(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    node = soup.find(id="single_law_section")
    if node:
        text = node.get_text("\n", strip=True)
    else:
        text = soup.get_text("\n", strip=True)[:50_000]
    return title_txt, text


def download_ins_codes() -> dict[str, int]:
    ids_path = OUT_DIR / "_california_ins_section_ids.txt"
    extras_path = OUT_DIR / "_extras_sections.txt"
    if not extras_path.exists():
        extras_path.write_text(
            "# Optional: one INS section number per line (no INS prefix).\n"
            "# Example:\n# 790.03\n# 1871.7\n",
            encoding="utf-8",
        )

    if REUSE_DISCOVERED_IDS and ids_path.exists() and ids_path.stat().st_size > 5:
        section_ids = {ln.strip().rstrip(".") for ln in ids_path.read_text(encoding="utf-8").splitlines() if ln.strip()}
        print(f"Loaded {len(section_ids)} section ids from {ids_path.name} (skipped Justia crawl)")
    else:
        section_ids = discover_section_ids_justia()
        ids_path.write_text("\n".join(sorted(section_ids, key=section_sort_key)), encoding="utf-8")

    section_ids |= load_extras(extras_path)
    ordered = sorted(section_ids, key=section_sort_key)
    print(f"Total section ids after extras: {len(ordered)}")

    todo = ordered if not MAX_SECTIONS else ordered[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote, skipped, failed = 0, 0, 0
    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        verify=_verify,
        http2=False,
    ) as client:
        for i, sid in enumerate(todo, 1):
            dest = OUT_DIR / section_id_to_filename(sid)
            if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
                skipped += 1
            else:
                url = leginfo_display_url(sid)
                try:
                    r = client.get(url, follow_redirects=True)
                    r.raise_for_status()
                    time.sleep(REQUEST_DELAY_SEC)
                    head_t, body_t = extract_leginfo_body(r.text)
                    title = head_t or f"California Insurance Code — Section {sid}"
                    md = (
                        f"# {title}\n\n"
                        f"**California Insurance Code (INS)**\n\n"
                        f"**Official source:** {url}\n\n"
                        f"**Section:** {sid}\n\n"
                        f"**Justia mirror (discovery):** {JUSTIA_PREFIX}\n\n"
                        f"---\n\n"
                        f"{body_t}\n"
                    )
                    dest.write_text(md, encoding="utf-8")
                    wrote += 1
                except Exception as e:
                    print(f"FAIL {sid}: {e}", flush=True)
                    failed += 1
            if i % 200 == 0:
                print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})", flush=True)

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_ins_codes()


Justia crawl: visited 2796 pages, found 2202 section ids
Total section ids after extras: 2202
… 200/2202 (wrote=200 skipped=0 failed=0)
… 400/2202 (wrote=400 skipped=0 failed=0)
… 600/2202 (wrote=600 skipped=0 failed=0)
… 800/2202 (wrote=800 skipped=0 failed=0)
… 1000/2202 (wrote=1000 skipped=0 failed=0)
… 1200/2202 (wrote=1200 skipped=0 failed=0)
… 1400/2202 (wrote=1400 skipped=0 failed=0)
… 1600/2202 (wrote=1600 skipped=0 failed=0)
… 1800/2202 (wrote=1800 skipped=0 failed=0)
… 2000/2202 (wrote=2000 skipped=0 failed=0)
… 2200/2202 (wrote=2200 skipped=0 failed=0)
Done. wrote=2202 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/california/ins_codes


{'wrote': 2202, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the repository root.
